In [ ]:
import pandas as pd
import numpy as np
import os
import yaml

# claude --resume 1066c47a-c343-4f01-89fa-4817a778ca26

In [48]:
L1_ORDER = {"Background": 0, "White Matter": 1, "Gray Matter": 2, "Cerebellum White Matter": 3, "Cerebellum Gray Matter": 4, 
       "Deep Gray Matter": 5, "Brainstem": 6, "CSF": 7}


ontology_v0 = pd.read_csv("data/inputs/nextbrain_ontology_v1.csv")
ontology_v0_l1 = ontology_v0["L1_tissue"].unique()
unique_ids = ontology_v0["allen_id"].unique()
unique_ids = [int(i) for i in unique_ids]

print("L1 tissue in ontology_v0:", ontology_v0_l1)

NAMES_MAPPING = {
    "Background": "Background",
    "White Matter": "Cerebral WM",
    "Gray Matter": "Cerebral GM",
    "Cerebellum White Matter": "Cerebellar WM",
    "Cerebellum Gray Matter": "Cerebellar GM",
    "Deep Gray Matter": "Deep GM",
    "Brainstem": "Brainstem",
    "CSF": "CSF / ventricles"
}


mapping_dict = {}
for key, value in NAMES_MAPPING.items():
    ids_list = ontology_v0[ontology_v0["L1_tissue"] == value]["allen_id"].to_list()
    # convert to int
    ids_list = [int(i) for i in ids_list]
    mapping_dict[key] = ids_list

mapped_ids = [id for sublist in mapping_dict.values() for id in sublist]

# print missing IDs in unique_ids that are not in mapped_ids
missing_ids = [id for id in unique_ids if id not in mapped_ids]
print("Missing IDs in mapping_dict:", missing_ids)


# Structures that are Other in the ontology 
mapping_dict["Brainstem"].append(506)  # Brainstem

# Node structures like the whole cortex
mapping_dict["Gray Matter"].append(3)  # Brainstem

# print the number of IDs in the mapping_dict
total_ids = sum([len(value) for value in mapping_dict.values()])
print(f"Matched ids: {total_ids} = {len(ontology_v0['allen_id'].unique())} (mapped ids vs unique ids in ontology_v1) if 3 is added left side should differ by 1")


# print("Mapping dict:", mapping_dict)
mapping_l1_path = "data/outputs/l1_names_mapping.yaml"
# save as yaml filliwing the order of the keys in L1_ORDER
with open(mapping_l1_path, "w") as f:
    yaml.dump(mapping_dict, f, sort_keys=False)

L1 tissue in ontology_v0: <StringArray>
[      'Background',        'Brainstem', 'CSF / ventricles',
    'Cerebellar GM',    'Cerebellar WM',      'Cerebral GM',
      'Cerebral WM',          'Deep GM',            'Other']
Length: 9, dtype: str
Missing IDs in mapping_dict: [506]
Matched ids: 336 = 335 (mapped ids vs unique ids in ontology_v1) if 3 is added left side should differ by 1


### FInd incoherences with converted_16l.yaml

In [ ]:
converted_16l_path = "data/inputs/converted_16l.yaml"

# load the converted_16l.yaml and flatten the list of single-key dicts into one dict
with open(converted_16l_path, "r") as f:
    raw = yaml.safe_load(f)
converted_16l = {k: [int(i) for i in v] for d in raw["Combined Structures"] for k, v in d.items()}

# id -> structure lookup for both dictionaries
id_to_mapping = {i: k for k, ids in mapping_dict.items() for i in ids}
id_to_converted = {i: k for k, ids in converted_16l.items() for i in ids}

# structures present in only one of the dictionaries
print("Structures only in mapping_dict:", set(mapping_dict) - set(converted_16l))
print("Structures only in converted_16l:", set(converted_16l) - set(mapping_dict))

# per-structure differences
for key in L1_ORDER:
    m = set(mapping_dict.get(key, []))
    c = set(converted_16l.get(key, []))
    if m == c:
        print(f"\n[OK] {key}: {len(m)} ids match")
        continue
    print(f"\n[MISMATCH] {key}: mapping_dict={len(m)} ids, converted_16l={len(c)} ids")
    only_m = sorted(m - c)
    only_c = sorted(c - m)
    if only_m:
        print("  only in mapping_dict :", [(i, f"converted_16l -> {id_to_converted.get(i, 'MISSING')}") for i in only_m])
    if only_c:
        print("  only in converted_16l:", [(i, f"mapping_dict -> {id_to_mapping.get(i, 'MISSING')}") for i in only_c])